# Exercise Sheet for Interpolation and Approximation
----------------------------------------------------------

In [ ]:
import numpy as np                
import matplotlib.pyplot as plt 

: 

## Question 1
### a) Compute via interpolation and plot the Lagrange polynomial p(x) to $f(x) = cos(-2\pi x) + 0.4x^2$ over
### $(x_i,y_i) = [(7.2,21.04),(7.4,21.12),(7.6,22.39),(7.8,24.77]$ 


### b) Estimate the maximum absolute error bound for this Lagrange polynomial approximation over $\xi \in [x_0,x_n]$

In [ ]:
x = np.arange(7,8,0.001)   
y = np.cos(-2*np.pi*x) +0.4*x**2  


In [ ]:
xi = np.array((7.2,7.4,7.6,7.8))  
yi = y[np.digitize(xi,x)]


In [ ]:
plt.figure(figsize=(10,10))
plt.plot(x,y,label='f(x)')  
plt.plot(xi,yi,'o',markersize=10)
plt.xlabel('x',fontsize=17)
plt.ylabel('y',fontsize=17)
plt.legend(loc='best',fontsize=15)


### Solution 1a – Lagrange polynomial

$$p(x)=\sum_{k=0}^{n} y_k\,L_k(x),\qquad L_k(x)=\prod_{m\neq k}\frac{x-x_m}{x_k-x_m}$$

**Note:** The $y_i$ given in the task text (21.12, 22.39, 24.77) do not match $f(x_i)$ exactly (e.g. $f(7.4)=21.095$). As in the provided code, I use the exact function values $y_i=f(x_i)$; the polynomial through the rounded task values is plotted for comparison.

In [ ]:
def f(x):
    return np.cos(-2*np.pi*x) + 0.4*x**2

def lagrange(x, xi, yi):
    
    x = np.asarray(x, dtype=float)
    p = np.zeros_like(x)
    for k in range(len(xi)):
        Lk = np.ones_like(x)
        for m in range(len(xi)):
            if m != k:
                Lk *= (x - xi[m]) / (xi[k] - xi[m])
        p += yi[k] * Lk
    return p

xi = np.array([7.2, 7.4, 7.6, 7.8])
yi = f(xi)                                       
yi_task = np.array([21.04, 21.12, 22.39, 24.77]) 

x = np.linspace(7.0, 8.0, 2001)
p = lagrange(x, xi, yi)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)
ax1.plot(x, f(x), label='f(x)')
ax1.plot(x, p, '--', label='Lagrange p(x), $y_i=f(x_i)$')
ax1.plot(x, lagrange(x, xi, yi_task), ':', label='Lagrange p(x), $y_i$ from task text')
ax1.plot(xi, yi, 'o', markersize=10, c='black')
ax1.axvspan(xi[0], xi[-1], color='grey', alpha=0.1)
ax1.set_ylabel('y', fontsize=17)
ax1.legend(loc='best', fontsize=13)

ax2.plot(x, np.abs(p - f(x)))
ax2.axvspan(xi[0], xi[-1], color='grey', alpha=0.1)
ax2.set_xlabel('x', fontsize=17)
ax2.set_ylabel('|p(x) - f(x)|', fontsize=17)
ax2.set_yscale('log')
plt.show()

print('y_i = f(x_i) =', np.round(yi, 4))

### Solution 1b – Error bound

$$|f(x)-p(x)| \le \frac{\max_{\xi\in[x_0,x_n]}|f^{(n+1)}(\xi)|}{(n+1)!}\;\max_{x\in[x_0,x_n]}\Big|\prod_{i=0}^{n}(x-x_i)\Big|$$

With $n=3$ (4 points) we need $f^{(4)}$. The $0.4x^2$ term vanishes after two derivatives, so
$$f^{(4)}(x) = (2\pi)^4\cos(2\pi x)\;\Rightarrow\;\max|f^{(4)}| = (2\pi)^4 \approx 1558.5$$
(the maximum $|\cos|=1$ is actually reached in the interval, at $x=7.5$).

For equidistant nodes with $h=0.2$: $\max|\prod(x-x_i)| = h^4\max_{s\in[0,3]}|s(s-1)(s-2)(s-3)| = h^4\cdot 1 = 0.0016$.

$$E_{max} \le \frac{1558.5}{24}\cdot 0.0016 \approx 0.104$$

In [ ]:
xx = np.linspace(xi[0], xi[-1], 100001)
w_max  = np.max(np.abs(np.prod([xx - a for a in xi], axis=0)))
f4_max = (2*np.pi)**4
E_bound = f4_max / 24 * w_max      # (n+1)! = 4! = 24

E_true = np.max(np.abs(lagrange(xx, xi, yi) - f(xx)))
print(f'max|f^(4)|          = {f4_max:.2f}')
print(f'max|prod(x - x_i)|  = {w_max:.5f}')
print(f'error bound         = {E_bound:.4f}')
print(f'actual max error    = {E_true:.4f}   (< bound, as it must be)')

----------------------------------------------------------

## Question 2
### Assume linear interpolation with step size $\textbf{d}$ is used to approximate $y(x) = \pi x^7$
### What step size for linear interpolation do you need to have for an absolute error $E_{max} < 10^{-6}$?. You can assume  $x \in [0,2]$.

### Solution 2

Error of linear interpolation between two nodes $x_i, x_{i+1}=x_i+d$:
$$|E| \le \frac{\max|y''|}{2}\max_{x}|(x-x_i)(x-x_{i+1})| = \frac{\max|y''|}{2}\cdot\frac{d^2}{4} = \frac{d^2}{8}\max|y''|$$

$y''(x) = 42\pi x^5$, maximal on $[0,2]$ at $x=2$: $\max|y''| = 1344\pi \approx 4222.3$

$$\frac{d^2}{8}\,1344\pi < 10^{-6}\;\Rightarrow\; d < \sqrt{\frac{8\cdot10^{-6}}{1344\pi}} \approx 4.35\cdot10^{-5}$$

i.e. roughly $46\,000$ intervals on $[0,2]$ (with a uniform grid – the requirement is set by the steep end at $x=2$).

In [ ]:
y2max = 42*np.pi*2**5
d_max = np.sqrt(8e-6 / y2max)
print(f'max|y\'\'| = {y2max:.2f}')
print(f'd_max     = {d_max:.4e}')


d  = 0.99*d_max
xg = np.arange(0, 2 + d, d)
xf = np.linspace(0, 2, 2_000_001)
E  = np.max(np.abs(np.interp(xf, xg, np.pi*xg**7) - np.pi*xf**7))
print(f'max error with d = {d:.3e}:  {E:.3e}  (< 1e-6)')

----------------------------------------------------------

## Question 3
### When using Taylor polynomials to approximate $y = cos(-3 \pi x) + 3x$ over the range $x = [-0.25,0.25]$, about $x= 0$;  to what order $j$ would you need to compute the interpolating polynomial in order to keep the maximum absolute error smaller than $E_{j}(x) < 10^{-6}$?

In [ ]:
xi = np.arange(-0.25,1,0.01)
xdum = np.arange(-0.25,0.25,0.01)
x = np.array([-0.25,-0.125,0,0.125,0.25])
y = np.cos(-3*np.pi*x) + 3*x
ydum = np.cos(-3*np.pi*xdum) + 3*xdum
yi = np.cos(-3*np.pi*xi) + 3*xi
plt.figure(figsize=(10,10))
plt.plot(xdum,ydum)
plt.plot(xi,yi,'--')
plt.plot(x,y,'o',c='black')
plt.xlabel('x',fontsize=17)
plt.ylabel('y',fontsize=17)

### Solution 3

Taylor remainder about $a=0$:
$$|E_j(x)| = \frac{|f^{(j+1)}(u)|}{(j+1)!}|x|^{j+1},\qquad u\in(0,x)$$

For $j+1\ge 2$ the linear term $3x$ is gone, and every derivative of $\cos(3\pi x)$ is bounded by $(3\pi)^{j+1}$. With $|x|\le 0.25$:
$$E_j \le \frac{(3\pi\cdot 0.25)^{j+1}}{(j+1)!} = \frac{(3\pi/4)^{j+1}}{(j+1)!} < 10^{-6}$$

In [ ]:
from math import factorial

for j in range(1, 30):
    bound = (3*np.pi*0.25)**(j+1) / factorial(j+1)
    print(f'j = {j:2d}:  bound = {bound:.3e}')
    if bound < 1e-6:
        break
print(f'\n-> order j = {j} needed')

In [ ]:

def taylor(x, j):

    p = 3*x if j >= 1 else 0*x
    for k in range(0, j+1, 2):
        p = p + (-1)**(k//2) * (3*np.pi*x)**k / factorial(k)
    return p

xs = np.linspace(-0.25, 0.25, 5001)
F  = np.cos(-3*np.pi*xs) + 3*xs
for jj in [10, 12, 14]:
    print(f'j = {jj}: actual max error = {np.max(np.abs(taylor(xs, jj) - F)):.2e}')

plt.figure(figsize=(10, 6))
for jj in [2, 4, 6, 8, 10, 12, 14]:
    plt.semilogy(xs, np.abs(taylor(xs, jj) - F) + 1e-18, label=f'j = {jj}')
plt.axhline(1e-6, c='black', ls='--', label='$10^{-6}$')
plt.ylim(1e-12, 10)
plt.xlabel('x', fontsize=17); plt.ylabel('$|E_j(x)|$', fontsize=17)
plt.legend(); plt.show()

**Answer: $j = 14$.** Since the cosine has only even Taylor terms, $p_{12}=p_{13}$; its actual max error is $1.8\cdot10^{-6}$, still too large. $p_{14}$ gives $4\cdot10^{-8}$ (bound $2.9\cdot10^{-7}$).

----------------------------------------------------------

## Question 4
## Use bilinear interpolation to determine the value of the function $f(x,y) = z$  at the point $(x_r, y_r)$, given the below four data points $x_i, y_i$ with values $z_{i} = f(x_i,y_i)$


In [ ]:
xi = np.array([0,0,1,1])  #x_1,x_1,x_2,x_2
yi = np.array([0,1,1,0])  #y_1,y_2,y_2,y_1
zi = np.array([-0.7,0.3,2.4,2.6])  


xr = np.random.uniform()
yr = np.random.uniform()


plt.figure(figsize=(10,10))
plt.scatter(xi,yi,c=zi,s=170,marker='o',cmap='jet',edgecolor='black')
plt.colorbar(label='z = f(x,y)')
plt.xlabel('X',fontsize=17)
plt.ylabel('Y',fontsize=17)
plt.scatter(xr,yr,edgecolor='black',marker='s',s=290,c='white')
plt.text(0.05,0.05,'($x_1,y_1)$',fontsize=12)
plt.text(0.05,0.95,'($x_1,y_2)$',fontsize=12)
plt.text(0.9,0.95,'($x_2,y_2)$',fontsize=12)
plt.text(0.9,0.05,'($x_2,y_1)$',fontsize=12)
plt.text(xr,yr-0.1,'($x_r,y_r)$',fontsize=20)





### Solution 4 – Bilinear interpolation

On the unit square ($x_1=y_1=0$, $x_2=y_2=1$), first interpolate linearly in $x$ along $y_1$ and $y_2$, then in $y$:

$$f(x,y) \approx \frac{1}{(x_2-x_1)(y_2-y_1)}\Big[z_{11}(x_2-x)(y_2-y) + z_{21}(x-x_1)(y_2-y) + z_{12}(x_2-x)(y-y_1) + z_{22}(x-x_1)(y-y_1)\Big]$$

with $z_{11}=f(0,0)=-0.7,\; z_{12}=f(0,1)=0.3,\; z_{22}=f(1,1)=2.4,\; z_{21}=f(1,0)=2.6$. Here this simplifies to
$$f(x,y) = -0.7 + 3.3\,x + 1.0\,y - 1.2\,xy$$

In [ ]:
def bilinear(x, y, x1, x2, y1, y2, z11, z12, z21, z22):
    # z_ab = f(x_a, y_b)
    fy1 = ((x2 - x)*z11 + (x - x1)*z21) / (x2 - x1)   # along y = y1
    fy2 = ((x2 - x)*z12 + (x - x1)*z22) / (x2 - x1)   # along y = y2
    return ((y2 - y)*fy1 + (y - y1)*fy2) / (y2 - y1)


z11, z12, z22, z21 = zi
zr = bilinear(xr, yr, 0, 1, 0, 1, z11, z12, z21, z22)
print(f'(x_r, y_r) = ({xr:.4f}, {yr:.4f})  ->  z_r = f(x_r, y_r) = {zr:.4f}')


print('corners:', [bilinear(a, b, 0, 1, 0, 1, z11, z12, z21, z22) for a, b in zip(xi, yi)])


gx, gy = np.meshgrid(np.linspace(0, 1, 200), np.linspace(0, 1, 200))
gz = bilinear(gx, gy, 0, 1, 0, 1, z11, z12, z21, z22)
plt.figure(figsize=(10, 9))
plt.pcolormesh(gx, gy, gz, cmap='jet', vmin=zi.min(), vmax=zi.max(), shading='auto')
plt.colorbar(label='z = f(x,y)')
plt.scatter(xi, yi, c=zi, s=170, cmap='jet', vmin=zi.min(), vmax=zi.max(), edgecolor='black')
plt.scatter(xr, yr, c=[zr], s=290, marker='s', cmap='jet', vmin=zi.min(), vmax=zi.max(), edgecolor='white')
plt.text(xr, yr - 0.07, f'$z_r$ = {zr:.3f}', fontsize=16, color='white')
plt.xlabel('X', fontsize=17); plt.ylabel('Y', fontsize=17)
plt.show()